# 12. Reproducibilidad y procedencia

El libro se reconstruye desde los datos del repositorio y conserva las salidas de cada análisis. El entorno documental se separa del entrenamiento. Las rutas se resuelven a partir de la raíz del repositorio y las fechas se interpretan en America/Bogota.

In [1]:
from pathlib import Path
import sys,platform,json,hashlib,importlib.metadata as metadata
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/processed/panel_horario.parquet').exists())
paquetes=['jupyter-book','nbformat','ipython','matplotlib','pandas','pyarrow','scipy','statsmodels','scikit-learn']
entorno={'python':sys.version,'sistema':platform.platform(),
    'dependencias':{p:metadata.version(p) for p in paquetes},'dispositivo_eda':'CPU',
    'zona_horaria':'America/Bogota'}
(ROOT/'results/jbook/entorno.json').write_text(json.dumps(entorno,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.Series(entorno['dependencias'],name='versión documental'))


jupyter-book     1.0.4
nbformat        5.11.1
ipython         9.18.1
matplotlib      3.11.2
pandas           3.1.1
pyarrow         26.0.1
scipy           1.18.1
statsmodels     0.15.0
scikit-learn     1.9.1
Name: versión documental, dtype: str

## Manifiesto de entradas del análisis

Las huellas SHA-256 identifican el contenido binario exacto. Se incluyen fuentes numéricas, titulares, archivos externos y el panel consolidado; los resultados de modelos se identifican por separado mediante su registro de artefactos. Calcular una huella no deserializa ni evalúa etiquetas de prueba.

In [2]:
def sha256_archivo(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for bloque in iter(lambda:f.read(1024*1024),b''):
            h.update(bloque)
    return h.hexdigest()

rutas=set(p for p in (ROOT/'data/raw/numeric').rglob('*') if p.is_file())
rutas.update(p for p in (ROOT/'data/external').rglob('*') if p.is_file())
rutas.update([ROOT/'data/raw/noticias/titulares.parquet',ROOT/'data/processed/panel_horario.parquet',
    ROOT/'data/processed/feature_registry.json',ROOT/'data/processed/diccionario_datos.json'])
manifiesto=pd.DataFrame([{'archivo':p.relative_to(ROOT).as_posix(),'bytes':p.stat().st_size,
    'sha256':sha256_archivo(p)} for p in sorted(rutas)])
manifiesto.to_csv(ROOT/'results/jbook/manifiesto_entradas.csv',index=False)
n_archivos=len(manifiesto); total_gb=float(manifiesto['bytes'].sum()/1e9)


El manifiesto registra 59 archivos y 0.06 GB de entradas. Sus huellas permiten detectar cambios de versión antes de reconstruir el análisis. El diccionario y el registro de disponibilidad complementan la identidad binaria con las definiciones semánticas y temporales.

## Ejecución en Windows y Linux

Crear un entorno con la versión de Python registrada y usar el archivo de dependencias fijadas. Los comandos siguientes se ejecutan desde la raíz; la activación del entorno depende del sistema operativo.

### Windows

```powershell
python -m venv .bookenv
.bookenv\Scripts\python.exe -m pip install -r requirements-book.lock.txt
.bookenv\Scripts\python.exe scripts\crear_jbook.py
.bookenv\Scripts\jupyter-book.exe build book --warningiserror --keep-going
.bookenv\Scripts\python.exe scripts\verificar_jbook.py
```

### Linux

```bash
python3 -m venv .bookenv
.bookenv/bin/python -m pip install -r requirements-book.lock.txt
.bookenv/bin/python scripts/crear_jbook.py
.bookenv/bin/jupyter-book build book --warningiserror --keep-going
.bookenv/bin/python scripts/verificar_jbook.py
```

Los análisis se ejecutan por CPU. El entrenamiento conserva sus dependencias en requirements.txt y permite seleccionar dispositivo mediante la interfaz del repositorio: --dispositivo auto, cpu o cuda. La GPU requiere PyTorch compatible con CUDA o ROCm y con su controlador; el entorno documental no lo exige. La compatibilidad de rutas no equivale a una ejecución verificada en ambos sistemas: el entorno efectivamente utilizado queda registrado en entorno.json.

## Conservación y reconstrucción de la entrega

El verificador comprueba estructura de cuadernos, celdas ejecutadas, ausencia de errores, interpretaciones junto a las figuras, cobertura del diccionario y enlaces HTML locales. Después copia el HTML construido a docs. No publica ni inicia entrenamiento. Los archivos del libro y sus evidencias se conservan en book, docs y results/jbook; una actualización del sitio requiere una solicitud expresa.

## Procedencia de modelos y reconstrucción

Los modelos se identifican mediante el commit d1ae9a4e09dd1dead4af1329f16aa3e8d0f2c3d3 y el [registro de artefactos](_static/modelos/modelos.json). El [índice SHA-256](_static/modelos/indice_artefactos_v3_v4.csv) registra 2.304 archivos y la selección conserva cinco checkpoints por familia. Las métricas se reconstruyen desde etiquetas y predicciones con idénticos orígenes. El código está en scripts/modelos_actualizados_jbook.py; el informe usa results/jbook/modelos_actuales.

Para reconstruir las secciones: python scripts/crear_jbook.py 08_literatura 09_validacion 10_modelos 10_evaluacion 11_sintesis 12_reproducibilidad; después, jupyter-book build book --warningiserror --keep-going y python scripts/verificar_jbook.py. Este procedimiento analiza artefactos guardados. El pipeline reproducible admite un dispositivo explícito y rutas con pathlib; la ejecución GPU registrada utiliza ROCm. La reproducción en Linux y Windows requiere instalar PyTorch compatible con el dispositivo elegido.

## Reconstrucción y revisión de consistencia

El generador integra modelos_actualizados_jbook.py, correcciones_e1_jbook.py y consistencia_jbook.py. La construcción del informe utiliza artefactos guardados; la medición de CPU se conserva en latencia_inferencia.csv y entorno_inferencia.json. Para repetir esa medición se utiliza un entorno separado con requirements-inferencia.lock.txt y scripts/medir_inferencia.py. No se ajustan pesos durante el análisis. La revisión global contrasta terminología, unidades, particiones, margen, cifras, inferencia, referencias y enlaces con los archivos fuente; sus resultados se registran en REVISION_CONSISTENCIA_JBOOK.md.